# merge_by_embedding benchmark

Evaluates `faiss_index` + threshold against `merge_test.json`.
Reports accuracy per section and prints wrong predictions.

In [1]:
import os, sys, json
sys.path.append(os.path.join(os.path.abspath(''), '..', 'backend'))

from embedding import compute_embeddings, faiss_index

/opt/anaconda3/envs/smartgraph/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
TEST_PATH = os.path.join(os.path.abspath(''), '..', 'data', 'test_data', 'merge_test.json')
THRESHOLD = 0.92

with open(TEST_PATH) as f:
    data = json.load(f)

sections = list(data.keys())
print(f'Sections: {sections}')
print(f'Threshold: {THRESHOLD}')

Sections: ['true_positives', 'true_negatives', 'soft_negatives']
Threshold: 0.92


In [3]:
# Collect all unique labels across all sections and embed once
all_labels = set()
for section in sections:
    for pair in data[section]:
        all_labels.add(pair['a'])
        all_labels.add(pair['b'])

all_labels = list(all_labels)
print(f'Embedding {len(all_labels)} unique labels...')
embeddings = compute_embeddings(all_labels)
print('Done.')

Embedding 341 unique labels...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6652.40it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Done.


In [4]:
def evaluate_section(pairs: list[dict], threshold: float) -> tuple[float, list[dict]]:
    """
    Evaluate merge predictions for a list of test pairs at a given threshold.

    For each pair, uses faiss_index to compute cosine similarity between the
    embeddings of label 'a' (document) and label 'b' (query). Predicts merge
    if similarity >= threshold and compares against the ground-truth 'should_merge' flag.

    Args:
        pairs:     List of dicts with keys 'a', 'b', 'should_merge'.
        threshold: Cosine similarity cutoff above which two labels are considered duplicates.

    Returns:
        accuracy: Fraction of pairs correctly classified (float).
        wrong:    List of dicts for misclassified pairs, each with keys
                  'a', 'b', 'similarity', 'should_merge'.
    """
    wrong = []
    correct = 0
    for pair in pairs:
        a, b, should_merge = pair['a'], pair['b'], pair['should_merge']
        _, _, sim = faiss_index(
            queries   = {b: embeddings[b]},
            documents = {a: embeddings[a]}
        )[0]
        if (sim >= threshold) == should_merge:
            correct += 1
        else:
            wrong.append({'a': a, 'b': b, 'similarity': round(sim, 4), 'should_merge': should_merge})
    return correct / len(pairs), wrong


results = {}
for section in sections:
    acc, wrong = evaluate_section(data[section], THRESHOLD)
    results[section] = {'accuracy': acc, 'wrong': wrong}

    print(f'\n── {section} ──')
    print(f'  Accuracy: {acc:.1%}  ({len(data[section]) - len(wrong)}/{len(data[section])} correct)')
    if wrong:
        print(f'  Wrong predictions ({len(wrong)}):')
        for w in wrong:
            direction = 'merged (should NOT)' if not w['should_merge'] else 'NOT merged (should)'
            print(f'    [{w["similarity"]:.4f}]  "{w["a"]}"  vs  "{w["b"]}"  ->  {direction}')
    else:
        print('  No wrong predictions.')


── true_positives ──
  Accuracy: 17.3%  (17/98 correct)
  Wrong predictions (81):
    [0.1248]  "BPE"  vs  "Byte Pair Encoding"  ->  NOT merged (should)
    [0.6017]  "KL Divergence"  vs  "KL"  ->  NOT merged (should)
    [0.6321]  "InfoNCE"  vs  "InfoNCE Loss"  ->  NOT merged (should)
    [0.2074]  "GPT"  vs  "Generative Pre-trained Transformer"  ->  NOT merged (should)
    [0.1560]  "BERT"  vs  "Bidirectional Encoder Representations from Transformers"  ->  NOT merged (should)
    [0.3072]  "ReLU"  vs  "Rectified Linear Unit"  ->  NOT merged (should)
    [0.6188]  "LSTM"  vs  "Long Short-Term Memory"  ->  NOT merged (should)
    [0.1899]  "GRU"  vs  "Gated Recurrent Unit"  ->  NOT merged (should)
    [0.5402]  "CNN"  vs  "Convolutional Neural Network"  ->  NOT merged (should)
    [0.2353]  "MLP"  vs  "Multi-Layer Perceptron"  ->  NOT merged (should)
    [0.8489]  "Cross Entropy"  vs  "Cross Entropy Loss"  ->  NOT merged (should)
    [0.8156]  "Transformer"  vs  "Transformer Model"  -

In [5]:
# Summary table
header = f"{'Section':<25} {'Accuracy':>10} {'Wrong':>8}"
print(header)
print('-' * len(header))
for section in sections:
    acc = results[section]['accuracy']
    n_wrong = len(results[section]['wrong'])
    print(f'{section:<25} {acc:>9.1%} {n_wrong:>8}')

Section                     Accuracy    Wrong
---------------------------------------------
true_positives                17.3%       81
true_negatives               100.0%        0
soft_negatives                99.0%        1


In [6]:
# Threshold sweep: find the best threshold across all sections
import numpy as np

thresholds = np.arange(0.70, 1.00, 0.02)
total_n = sum(len(data[s]) for s in sections)

header = f"{'Threshold':<12}" + "".join(f"{s:<25}" for s in sections) + "Overall"
print(header)
print('-' * len(header))

for t in thresholds:
    total_correct = 0
    row = f'{t:<12.2f}'
    for section in sections:
        acc, _ = evaluate_section(data[section], t)
        total_correct += int(acc * len(data[section]))
        row += f'{acc:<25.1%}'
    row += f'{total_correct / total_n:.1%}'
    print(row)

Threshold   true_positives           true_negatives           soft_negatives           Overall
----------------------------------------------------------------------------------------------
0.70        48.0%                    100.0%                   62.6%                    70.3%
0.72        45.9%                    100.0%                   67.7%                    71.3%
0.74        43.9%                    100.0%                   71.7%                    72.0%
0.76        42.9%                    100.0%                   74.7%                    72.6%
0.78        41.8%                    100.0%                   77.8%                    73.3%
0.80        39.8%                    100.0%                   81.8%                    74.0%
0.82        38.8%                    100.0%                   86.9%                    75.3%
0.84        35.7%                    100.0%                   87.9%                    74.7%
0.86        31.6%                    100.0%                   88.9